# CPU와 GPU

In [1]:
import torch

device = torch.device('cpu')
# device_cuda = torch.device('cuda')
# device_mps = torch.device('mps')

print(device)

cpu


In [4]:
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print(device)

mps


In [5]:
tensor = torch.randn(1, 2)
tensor_device = tensor.to(device)

print('텐서의 위치: ', tensor_device.device)

텐서의 위치:  mps:0


In [6]:
import torch.nn as nn

class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.fc1 = nn.Linear(28*28, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, 10)

    def forward(self, x):
        x = x.view(-1, 28*28)
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x)
        return x

model = Net().to(device)# 모델 GPU로 이동

In [7]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# 장치 사용 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 랜덤 데이터 및 레이블 생성
input_size = 100
hidden_size = 128
output_size = 10
num_samples = 1000

# 신경망 모델 정의
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.fc3 = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        return self.fc3(x)
    
features = torch.randn(num_samples, input_size)
labels = torch.randint(0, output_size, (num_samples,))

# TensorDataset과 DataLoader 생성
dataset = TensorDataset(features, labels)
train_loader = DataLoader(dataset=dataset, batch_size=64, shuffle=True)

# 디바이스를 지정해 모델 객체 생성
model = Net().to(device)

# 손실 함수 및 옵티마이저 설정
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 학습
model.train()
epochs = 20
for epoch in range(epochs):
    model.train()
    for inputs, labels in train_loader:
        # 데이터 GPU로 이동
        inputs, labels = inputs.to(device), labels.to(device)

        # 순전파
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        # 역전파 및 최적화
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
    print(f'Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}')

# 평가
model.eval()
with torch.no_grad():
    correct = 0
    total = 0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    print(f'Accuracy of the network on the 1000 random samples: {100 * correct / total}%')

Epoch [1/20], Loss: 2.3246
Epoch [2/20], Loss: 2.2680
Epoch [3/20], Loss: 2.2000
Epoch [4/20], Loss: 2.1178
Epoch [5/20], Loss: 2.0646
Epoch [6/20], Loss: 2.0244
Epoch [7/20], Loss: 1.7680
Epoch [8/20], Loss: 1.7233
Epoch [9/20], Loss: 1.3839
Epoch [10/20], Loss: 1.4125
Epoch [11/20], Loss: 1.4380
Epoch [12/20], Loss: 1.0917
Epoch [13/20], Loss: 0.7335
Epoch [14/20], Loss: 0.8249
Epoch [15/20], Loss: 0.6210
Epoch [16/20], Loss: 0.4822
Epoch [17/20], Loss: 0.4582
Epoch [18/20], Loss: 0.3145
Epoch [19/20], Loss: 0.2860
Epoch [20/20], Loss: 0.1436
Accuracy of the network on the 1000 random samples: 99.9%
